# Preliminari

Si impostano directory di lavoro e si fanno import per spark

In [1]:
import os
from pyspark.sql import SparkSession

DATASETS_DIR = "../dataset/"

spark = (
    SparkSession.builder
    .appName("pfp")
    .getOrCreate()
)

sc = spark.sparkContext


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/04 14:15:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Pre-Processing


Creo un dataframe dal file .parquet di input.

Creo i record (rdd) come necessario dal problema, ovvero chiave dell'ordine e valore le tuple contenente id oggetto e quantità.
Questi record li chiameremo Transazioni, come suggerito dal paper PFP.

In [2]:
sdf = spark.read.parquet(
    os.path.join(DATASETS_DIR, "online_retail.parquet")
)

transactions = (
    sdf
    .select("InvoiceNo", "StockCode", "Quantity")
    .rdd
    .map(lambda row: (row["InvoiceNo"], (row["StockCode"], row["Quantity"])))
    .groupByKey()
    .mapValues(list)
)
transactions.take(5)

[('536365',
  [('85123A', '6'),
   ('71053', '6'),
   ('84406B', '8'),
   ('84029G', '6'),
   ('84029E', '6'),
   ('22752', '2'),
   ('21730', '6')]),
 ('536366', [('22633', '6'), ('22632', '6')]),
 ('536367',
  [('84879', '32'),
   ('22745', '6'),
   ('22748', '6'),
   ('22749', '8'),
   ('22310', '6'),
   ('84969', '6'),
   ('22623', '3'),
   ('22622', '2'),
   ('21754', '3'),
   ('21755', '3'),
   ('21777', '4'),
   ('48187', '4')]),
 ('536368', [('22960', '6'), ('22913', '3'), ('22912', '3'), ('22914', '3')]),
 ('536369', [('21756', '3')])]

### Conversione

convertiamo gli oggetti (tuple chiave e quantità) in nuovi oggetti identificati da un numero, in questo modo si può facilmente utilizzare PFP con le quantità.
Riduciamo funzionalmente il problema di tenere in considerazione le quantità al problema senza le quantità per poi tornare al problema delle quantità.

T' = T
for t in T'
    t -> t'

out = PFP(T')

reversed = revert(out)

return alpha_code(reversed)

Per fare tutto ciò innanzitutto devo prendere gli oggetti e quantità e mapparli:

In [3]:
pairs= (
    transactions
    .flatMap(lambda x: x[1])                 # prendo tutte le tuple
    .distinct()                              # tuple uniche
    .sortBy(lambda pair: (pair[0], pair[1])) # ordine stabile
    .zipWithIndex()                          # assegna indice 0,1,2...
    .map(lambda x: (x[0], x[1] + 1))         # ((code, quantity), id)
)

print("ci sono " + str(pairs.count()) + " coppie")
pairs.take(50)


ci sono 45280 coppie


[(('10002', '-3'), 1),
 (('10002', '1'), 2),
 (('10002', '10'), 3),
 (('10002', '11'), 4),
 (('10002', '12'), 5),
 (('10002', '120'), 6),
 (('10002', '14'), 7),
 (('10002', '18'), 8),
 (('10002', '180'), 9),
 (('10002', '2'), 10),
 (('10002', '24'), 11),
 (('10002', '3'), 12),
 (('10002', '36'), 13),
 (('10002', '4'), 14),
 (('10002', '48'), 15),
 (('10002', '5'), 16),
 (('10002', '6'), 17),
 (('10002', '60'), 18),
 (('10002', '62'), 19),
 (('10002', '8'), 20),
 (('10080', '1'), 21),
 (('10080', '12'), 22),
 (('10080', '170'), 23),
 (('10080', '2'), 24),
 (('10080', '22'), 25),
 (('10080', '24'), 26),
 (('10080', '26'), 27),
 (('10080', '3'), 28),
 (('10080', '4'), 29),
 (('10080', '48'), 30),
 (('10120', '1'), 31),
 (('10120', '10'), 32),
 (('10120', '11'), 33),
 (('10120', '12'), 34),
 (('10120', '2'), 35),
 (('10120', '20'), 36),
 (('10120', '3'), 37),
 (('10120', '30'), 38),
 (('10120', '4'), 39),
 (('10120', '5'), 40),
 (('10120', '6'), 41),
 (('10120', '8'), 42),
 (('10123C', '-1

In [4]:
# Creo la mappa di conversione da tupla a numero
conversion_map = pairs.collectAsMap()
# Lo distribuisco ai worker in broadcast
bc_map = sc.broadcast(conversion_map)

# Rimappo ogni transazione
transactions_ids = transactions.mapValues(
    lambda items: [bc_map.value[item] for item in items]
)
print(transactions_ids.count())
transactions_ids.take(5)

25900


[('536365', [42871, 36544, 38857, 38275, 38242, 23017, 9285]),
 ('536366', [21178, 21148]),
 ('536367',
  [40566,
   22914,
   22959,
   22976,
   16064,
   41171,
   20968,
   20952,
   9514,
   9531,
   9632,
   36265]),
 ('536368', [25907, 25162, 25152, 25172]),
 ('536369', [9544])]

## PFP

Iniziamo ad implementare **PFP**, definiamo una variabile **epsilon** che rappresenta la "predefined minimum support threshold"
soglia minima predefinita di supporto.
Quindi una threshold sopra la quale verrà riconosciuto un pattern e i pattern sotto questa soglia verranno scartati 

In [ ]:
epsilon = 100

item_counts = (
    transactions_ids
    .flatMap(lambda row: set(row[1]))   # ogni item contato una sola volta per transazione
    .map(lambda item: (item, 1))
    .reduceByKey(lambda a, b: a + b)
    .filter(lambda row: row[1] >= epsilon)
)
print(item_counts.count())
item_counts.take(10)


212


[(42871, 560),
 (9514, 269),
 (9531, 201),
 (25907, 451),
 (18813, 249),
 (37859, 372),
 (37779, 256),
 (37756, 413),
 (11787, 327),
 (11427, 340)]

Creo la F-List che è la lista decrescente degli item (item = id_of(tuple(code, quantity)))

Poi ordino le transazione per "supporto" ovvero in base ai valori di F-List

Creo infine la Q-List tramite la quale si suddivide il calcolo tra le macchine.

In [8]:
#creo f_list
f_list = item_counts.sortBy(
    lambda row: (-row[1], row[0])
)

f_list.take(10)

# Creo una mappa per sortare le transazioni, la mappa è fatta così:  item_id -> posizione nella F-list

# Base comune: item_id -> rank nella F-list
item_rank = (
    f_list
    .map(lambda row: row[0])      # item_id
    .zipWithIndex()               # item_id -> rank
    .map(lambda x: (x[0], int(x[1])))
    .persist()
)

f_rank = item_rank.collectAsMap()
# notifico i worker
bc_f_rank = sc.broadcast(f_rank)



ordered_transactions = (
    transactions_ids
    .mapValues(
        lambda items: sorted(
            # tieni item solo se item è una chiave del dizionario f_rank
            set(item for item in items if item in bc_f_rank.value),
            key=lambda item: bc_f_rank.value[item]
        )
    )
    .filter(lambda row: len(row[1]) > 0)
)
print(ordered_transactions.count())
print(ordered_transactions.take(5))


# G-List
Q = 20

g_list = (
    item_rank
    .map(lambda x: (x[0], int(x[1] % Q)))   # item_id -> gid
    .collectAsMap()
)

bc_g_list = sc.broadcast(g_list)

13683
[('536365', [42871]), ('536367', [9514, 9531]), ('536368', [25907]), ('536370', [18813]), ('536373', [42871, 37756, 37859, 37779])]


In [9]:
def generate_group_dependent_transactions(row):
    invoice_no, items = row

    output = []
    seen_gids = set()

    # Scorro la transazione da destra verso sinistra
    for j in range(len(items) - 1, -1, -1):
        item = items[j]
        gid = bc_g_list.value.get(item)

        # Se questo gruppo non è ancora stato emesso per questa transazione
        if gid is not None and gid not in seen_gids:
            seen_gids.add(gid)

            # Emetto il prefisso fino alla posizione j inclusa
            output.append((gid, items[:j + 1]))

    return output

group_dependent_transactions = ordered_transactions.flatMap(
    generate_group_dependent_transactions
)

group_dependent_transactions.take(10)

[(7, [42871]),
 (9, [9514, 9531]),
 (0, [9514]),
 (17, [25907]),
 (13, [18813]),
 (5, [42871, 37756, 37859, 37779]),
 (13, [42871, 37756, 37859]),
 (7, [42871]),
 (5, [42871, 37756, 37859, 37779]),
 (13, [42871, 37756, 37859])]